In [1]:
import pandas as pd
import tensorflow as tf
import numpy as np
import copy
import random

2025-12-03 21:56:00.683674: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-12-03 21:56:00.683729: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-12-03 21:56:00.685107: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-12-03 21:56:00.694071: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-03 21:56:01.779957: W tensorflow/compiler/tf2

In [2]:
batch_size = 128
learning_rate = 0.001

In [3]:
@tf.keras.saving.register_keras_serializable()
class MLP(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.dense1 = tf.keras.layers.Dense(units=512, activation=tf.nn.leaky_relu)
        self.dense2 = tf.keras.layers.Dense(units=1024, activation=tf.nn.leaky_relu)
        self.dense3 = tf.keras.layers.Dense(units=512, activation=tf.nn.leaky_relu)
        self.dense4 = tf.keras.layers.Dense(units=256, activation=tf.nn.leaky_relu)
        self.dense5 = tf.keras.layers.Dense(units=8)

    def call(self, inputs):
        x = self.dense1(inputs)
        x = self.dense2(x)
        x = self.dense3(x)
        x = self.dense4(x)
        output = self.dense5(x)
        return output

In [4]:
class ParaServer:
    def __init__(self):
        self.model = MLP()
        self.optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    def upload(self, grads):
        self.optimizer.apply_gradients(grads_and_vars=zip(grads, self.model.variables))
        return self.model
    def download(self):
        return self.model
    def initModel(self, x):
        self.model(x)

In [5]:
def valiAll(index_epoch):
    m = ps.download()
    model = copy.deepcopy(m)
    y_v_p = model(X_v)
    va_mse = tf.reduce_mean(tf.square(y_v_p - y_v))
    va_rmse = tf.sqrt(va_mse)
    va_mae = tf.reduce_mean(tf.abs(y_v_p - y_v))
    va_r2 = 1 - tf.reduce_sum(tf.square(y_v_p - y_v)) / tf.reduce_sum(tf.square(y_v - tf.reduce_mean(y_v)))
    print("mse:{} rmse:{} mae:{} r2:{}".format(va_mse, va_rmse, va_mae, va_r2))
    r2sv[index_epoch] = va_r2.numpy()

In [6]:
class Node:
    def __init__(self, dsName, freq, mu=1e-4):
        self.freq = freq
        self.model = MLP()
        self.mu = mu
        self.dataset = pd.read_csv(dsName, encoding='utf-8').sample(frac=1).reset_index(drop=True)
        self.X = self.dataset.loc[:,'freq':'L4'].to_numpy(dtype = np.float32)
        self.y = self.dataset.loc[:,'S11r':'S41i'].to_numpy(dtype = np.float32)
        self.dataset_train = tf.data.Dataset.from_tensor_slices((self.X, self.y))
        self.dataset_train = self.dataset_train.shuffle(buffer_size=23000)
        self.dataset_train = self.dataset_train.batch(batch_size)
        self.dataset_train = self.dataset_train.prefetch(tf.data.experimental.AUTOTUNE)
    def train(self, index_epoch):
        m = ps.download()
        self.model = copy.deepcopy(m)
        global_weights = [tf.identity(w) for w in self.model.trainable_variables]
        for X, y in self.dataset_train:
            with tf.GradientTape() as tape:
                y_pred = self.model(X)
                tr_mse = tf.reduce_mean(tf.square(y_pred - y))
                prox_term = tf.add_n([
                        tf.nn.l2_loss(w - w0)
                        for w, w0 in zip(self.model.trainable_variables, global_weights)
                    ])
                loss = tr_mse + self.mu * prox_term
            tr_rmse = tf.sqrt(tr_mse)
            tr_mae = tf.reduce_mean(tf.abs(y_pred - y))
            tr_r2 = 1 - tf.reduce_sum(tf.square(y_pred - y)) / tf.reduce_sum(tf.square(y - tf.reduce_mean(y)))
            grads = tape.gradient(loss, self.model.variables)
            m = ps.upload(grads)
            self.model = copy.deepcopy(m)
        # if epoch_index in np.arange(0, num_epochs, 25).tolist() or epoch_index == num_epochs - 1:
        if True:
            print("node:{} epoch:{}".format(self.freq, index_epoch))
            print("train mse:{} rmse:{} mae:{} r2:{}".format(tr_mse, tr_rmse, tr_mae, tr_r2))
            r2s[self.freq][index_epoch] = tr_r2.numpy()

In [7]:
r2s = {2.4:{},2.5:{},2.6:{}}
r2sv = {}

In [8]:
test_dataset = pd.read_csv("Test.csv", encoding='utf-8').sample(frac=1).reset_index(drop=True)
X_v = test_dataset.loc[:,'freq':'L4'].to_numpy(dtype = np.float32)
y_v = test_dataset.loc[:,'S11r':'S41i'].to_numpy(dtype = np.float32)

In [9]:
ps = ParaServer()
ps.initModel(X_v)

2025-12-03 21:56:02.881113: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 900 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:02:00.0, compute capability: 8.9


In [10]:
nodeList = [Node('./24Train.csv', 2.4), Node('./25Train.csv', 2.5), Node('./26Train.csv', 2.6)]

In [11]:
orders = [0, 1, 2]
turn = [np.array([[26, 104], [178, 312], [344, 464], [520, 600]]), np.array([[0, 94], [149, 223], [319, 433], [464, 580]]), np.array([[32, 151], [155, 248], [270, 354], [378, 502]])]
for i in range(600):
    random.shuffle(orders)
    for j in orders:
        for l, r in turn[j]:
            if l <= i < r:
                nodeList[j].train(i)
    valiAll(i)

2025-12-03 21:56:04.100978: I external/local_xla/xla/service/service.cc:168] XLA service 0xd2d0300 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-12-03 21:56:04.101016: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 4090, Compute Capability 8.9
2025-12-03 21:56:04.108385: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-12-03 21:56:04.130852: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907
I0000 00:00:1764798964.279388    9261 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


node:2.5 epoch:0
train mse:0.08591631799936295 rmse:0.29311487078666687 mae:0.235477477312088 r2:0.2895238399505615
mse:0.09437987208366394 rmse:0.30721306800842285 mae:0.2464514672756195 r2:0.2187345027923584
node:2.5 epoch:1
train mse:0.08196600526571274 rmse:0.2862970530986786 mae:0.23069345951080322 r2:0.32603919506073
mse:0.08961062133312225 rmse:0.2993503212928772 mae:0.24248093366622925 r2:0.25821375846862793
node:2.5 epoch:2
train mse:0.07320936769247055 rmse:0.270572304725647 mae:0.21863004565238953 r2:0.3865342140197754
mse:0.0794541984796524 rmse:0.28187620639801025 mae:0.22736519575119019 r2:0.34228748083114624
node:2.5 epoch:3
train mse:0.06815724074840546 rmse:0.2610694169998169 mae:0.2077222764492035 r2:0.44026613235473633
mse:0.07741128653287888 rmse:0.2782288193702698 mae:0.22205042839050293 r2:0.3591983914375305
node:2.5 epoch:4
train mse:0.06093915179371834 rmse:0.24685856699943542 mae:0.19603951275348663 r2:0.48979562520980835
mse:0.06810931861400604 rmse:0.26097762

In [14]:
for k, v in r2sv.items():
    print(v)

0.2187345
0.25821376
0.34228748
0.3591984
0.436199
0.43097526
0.4622025
0.46772504
0.4935174
0.5355185
0.5337026
0.57015616
0.5867343
0.5961914
0.62451565
0.56967807
0.628634
0.62512016
0.6417531
0.599571
0.6358283
0.6423821
0.62096506
0.5967265
0.632769
0.6473693
0.62833047
0.6416918
0.64691615
0.6545862
0.6530363
0.6555068
0.5337534
0.48544693
0.49330473
0.6610825
0.4622038
0.3824808
0.44391894
0.45611763
0.66268325
0.66377044
0.6661345
0.5118242
0.4297223
0.47938013
0.4372837
0.6633941
0.44380277
0.48776633
0.43003464
0.4538278
0.39857578
0.44492865
0.4294412
0.3714261
0.6702243
0.4556589
0.6741725
0.44022423
0.46386552
0.44409156
0.45872998
0.48042524
0.660221
0.6631558
0.4395097
0.39443833
0.48878956
0.39318848
0.4449455
0.6587944
0.6573484
0.4608562
0.4523353
0.664466
0.46700698
0.66248167
0.65875274
0.41674978
0.38007635
0.66056335
0.4176829
0.47893786
0.45064098
0.66773796
0.66559833
0.43518078
0.66227096
0.65369165
0.44971347
0.4072799
0.41011763
0.4002313
0.42468154
0.4025384